# Project 5 | Notebook 1: Text Cleaning & Preprocessing

## Overview

This notebook prepares the raw BRICS monetary policy statements dataset for 
sentiment analysis (Notebook 2) and LDA topic modelling (Notebook 3). The input 
is `04_central_bank_scraper/data/brics_mpc_statements_v2.csv` — 302 statements 
from four central banks.

**Note on RBI data:** The original scraped dataset contained only 6 duplicate RBI 
statements (2025–2026) due to a JavaScript-blocked archive on the RBI website. 
This was resolved by manually downloading 57 PDF statements (2016–2025) from 
the RBI archive and extracting text using PyMuPDF. These were merged with the 6 
scraped statements to produce a complete RBI dataset of 63 statements covering 
2016–2026. The merged dataset was saved as `brics_mpc_statements_v2.csv`.

---

## Dataset Summary

| Bank | Country | Statements | Coverage |
|------|---------|------------|----------|
| PBOC | China | 131 | 1996–2025 |
| RBI | India | 63 | 2016–2026 |
| SARB | South Africa | 57 | 2006–2026 |
| CBR | Russia | 51 | 2018–2026 |

**Columns:** `country`, `central_bank`, `date`, `title`, `text`, `url`  
**Nulls:** None  
**Date dtype:** All dates are stored as strings — will be parsed to datetime

---

## Known Data Quality Issues

### 1. RBI — Boilerplate Header Text
Scraped RBI statements (2025–2026) begin with a press release header before 
the actual content: `"Press Releases ( 434 kb ) Date : [date] [title]"`. 
PDF-extracted statements (2016–2025) begin with Hindi header text and press 
release metadata. Both are stripped so text begins at the actual statement body.

### 2. SARB — Boilerplate Header Block
SARB statements begin with a structured metadata block containing the current 
repo rate, inflation rate, next meeting dates, and tolerance band figures before 
the Governor's actual statement begins. This will be stripped up to the point 
where the substantive text starts.

### 3. SARB — Malformed Dates
Older SARB statements have date fields containing URL fragment IDs rather than 
proper dates. These will be corrected where possible — year is always recoverable, 
month is not for pre-2024 entries. Affected rows flagged with `date_approximate=True`.

### 4. PBOC — Double Spaces
PBOC text contains double (and sometimes triple) spaces between words, likely 
introduced during PDF-to-text conversion in the BIS data pipeline. These will 
be collapsed to single spaces.

---

## Cleaning Pipeline

**Step 1 — Inspect SARB dates**  
Identify malformed date values and apply targeted corrections.

**Step 2 — Strip RBI boilerplate**  
Remove press release headers from scraped RBI statements and Hindi/metadata 
headers from PDF-extracted statements.

**Step 3 — Strip SARB boilerplate**  
Remove metadata header block, retaining only the Governor's statement body.

**Step 4 — Fix PBOC spacing**  
Collapse multiple consecutive whitespace characters to single spaces.

**Step 5 — Parse dates**  
Convert all date strings to `datetime` objects using a flexible multi-format 
parser. RBI PDF statements use ISO format (YYYY-MM-DD).

**Step 6 — General text cleaning**  
Applied uniformly across all banks:
- Lowercase all text
- Remove URLs, markdown headers, standalone numbers
- Replace hyphens and slashes with spaces
- Remove punctuation and special characters
- Remove stopwords using spaCy's English stopword list plus custom 
  central banking terms
- Lemmatise tokens using spaCy's `en_core_web_sm` model
- Store cleaned text in `text_clean`, preserving original `text`

**Step 7 — Final checks and export**  
Verify row counts, check for nulls, and save to 
`05_monetary_policy_sentiment/data/brics_mpc_cleaned.csv`.

---

## Output

A cleaned CSV with all original columns retained, plus:
- `text_clean` — preprocessed text ready for sentiment scoring and LDA
- `token_count` — number of tokens in cleaned text
- `date_approximate` — `True` for 46 SARB rows where only year was recoverable
- `date_original` — original raw date string before parsing

In [1]:
# ─────────────────────────────────────────────
# Project 5 | Notebook 1: Text Cleaning & Preprocessing
# BRICS Monetary Policy Sentiment Analysis
# ─────────────────────────────────────────────

import pandas as pd
import numpy as np
import re
import spacy
from pathlib import Path

# Display settings
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', 20)

print("Libraries loaded successfully")
print(f"spaCy version: {spacy.__version__}")

Libraries loaded successfully
spaCy version: 3.8.11


/Users/psat0501/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Define paths relative to repo root
DATA_IN = Path("../04_central_bank_scraper/data/brics_mpc_statements_v2.csv")
DATA_OUT = Path("data/brics_mpc_cleaned.csv")

# Create output directory if it doesn't exist
DATA_OUT.parent.mkdir(parents=True, exist_ok=True)

# Load
df = pd.read_csv(DATA_IN)

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nRow counts by bank:")
print(df['central_bank'].value_counts())

Shape: (302, 6)

Columns: ['country', 'central_bank', 'date', 'title', 'text', 'url']

Row counts by bank:
central_bank
PBOC    131
RBI      63
SARB     57
CBR      51
Name: count, dtype: int64


In [3]:
# Check dtypes and nulls
print("=== Data Types ===")
print(df.dtypes)

print("\n=== Null Counts ===")
print(df.isnull().sum())

print("\n=== Sample rows (one per bank) ===")
for bank in df['central_bank'].unique():
    print(f"\n--- {bank} ---")
    print(df[df['central_bank'] == bank][['date', 'title', 'text']].iloc[0])

=== Data Types ===
country         object
central_bank    object
date            object
title           object
text            object
url             object
dtype: object

=== Null Counts ===
country         0
central_bank    0
date            0
title           0
text            0
url             0
dtype: int64

=== Sample rows (one per bank) ===

--- CBR ---
date                                                                                        13 February 2026
title                                               Bank of Russia cuts the key rate by 50 bp to 15.50% p.a.
text     On 13 February 2026, the Bank of Russia Board of Directors decided to cut the key rate by 50 bas...
Name: 0, dtype: object

--- SARB ---
date                                                                                            January 2026
title                                                Statement of the Monetary Policy Committee January 2026
text     Statement of the Monetary Policy Committee Fore

In [4]:
# Inspect one sample per bank in full
for bank in ['RBI', 'CBR', 'SARB', 'PBOC']:
    row = df[df['central_bank'] == bank].iloc[0]
    print(f"\n{'='*60}")
    print(f"BANK: {bank}")
    print(f"DATE: {row['date']}")
    print(f"TITLE: {row['title']}")
    print(f"TEXT (first 500 chars):\n{row['text'][:500]}")


BANK: RBI
DATE: 2016-04-05
TITLE: Governor's Statement: April 5, 2016
TEXT (first 500 chars):
प्रेस प्रकाशनी  PRESS RELEASE 
संचार ͪवभाग, कɅ द्रȣय कायार्लय, एस.बी.एस.मागर्, मुंबई-400001 
_____________________________________________________________________________________________________________________
DEPARTMENT OF COMMUNICATION, Central Office, S.B.S.Marg, Mumbai-400001 
फोन/Phone: 91 22 2266 0502 फैक्स/Fax: 91 22 22660358 
 
भारतीय ǐरज़वर् बɇक 
RESERVE BANK OF INDIA  
0वेबसाइट : www.rbi.org.in/hindi 
Website : www.rbi.org.in 
इ-मेल email: helpdoc@rbi.org.in 
April 05, 2016 
First Bi-m

BANK: CBR
DATE: 13 February 2026
TITLE: Bank of Russia cuts the key rate by 50 bp to 15.50% p.a.
TEXT (first 500 chars):
On 13 February 2026, the Bank of Russia Board of Directors decided to cut the key rate by 50 basis points to 15.50% per annum. The economy continues to return to a balanced growth path. In January, price growth accelerated significantly due to one-off factors. However, the Bank of 

In [5]:
# Inspect SARB dates — check for malformed ones
sarb_dates = df[df['central_bank'] == 'SARB']['date'].values
print("All SARB dates:\n")
for d in sarb_dates:
    print(d)

All SARB dates:

January 2026
November 2025
September 2025
July 2025
May 2025
March 2025
January 2025
Statement Of The Monetary Policy Committee September 20241 2024
Statement Of The Monetary Policy Committee September 2024 2024
Statement Of The Monetary Policy Committee July 2024 2024
Statement Of The Monetary Policy Committee May 2024 2024
5726 2013
5478 2013
5301 2012
5141 2012
5092 2012
5040 2012
5005 2012
4930 2012
4881 2011
4842 2011
4706 2011
4082 2011
3978 2011
16 2011
3566 2010
3565 2010
3564 2010
3563 2010
3562 2010
3561 2010
4046 2009
3507 2009
4045 2009
3494 2009
3448 2009
3436 2009
3416 2009
3406 2009
3389 2009
3374 2008
3348 2008
3323 2008
3300 2008
3268 2008
4043 2008
4285 2007
4286 2007
4287 2007
4288 2007
4289 2007
4290 2007
6600 2006
4292 2006
4293 2006
4294 2006
4295 2006


In [6]:
# Categorise SARB date formats
import re

def classify_sarb_date(d):
    d = str(d).strip()
    if re.match(r'^(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4}$', d):
        return 'clean'
    elif re.match(r'^\d{4,5}\s+\d{4}$', d):
        return 'numeric_id'
    elif 'Statement' in d:
        return 'garbled_title'
    else:
        return 'unknown'

sarb_df = df[df['central_bank'] == 'SARB'][['date', 'url']].copy()
sarb_df['date_format'] = sarb_df['date'].apply(classify_sarb_date)

print(sarb_df['date_format'].value_counts())
print("\n--- Garbled title examples ---")
print(sarb_df[sarb_df['date_format'] == 'garbled_title'][['date', 'url']].to_string())
print("\n--- Numeric ID examples (first 10) ---")
print(sarb_df[sarb_df['date_format'] == 'numeric_id'][['date', 'url']].head(10).to_string())

date_format
numeric_id       45
clean             7
garbled_title     4
unknown           1
Name: count, dtype: int64

--- Garbled title examples ---
                                                               date                                                                                                                                                                            url
58  Statement Of The Monetary Policy Committee September 20241 2024  https://www.resbank.co.za/en/home/publications/publication-detail-pages/statements/monetary-policy-statements/2024/Statement-of-the-Monetary-Policy-Committee-September-20241
59   Statement Of The Monetary Policy Committee September 2024 2024   https://www.resbank.co.za/en/home/publications/publication-detail-pages/statements/monetary-policy-statements/2024/Statement-of-the-Monetary-Policy-Committee-September-2024
60        Statement Of The Monetary Policy Committee July 2024 2024        https://www.resbank.co.za/en/home/publications

In [8]:
sarb_df[sarb_df['date_format'] == 'unknown'][['date', 'url']]

,date,url
75,16 2011,https://www.resbank.co.za/en/home/publications/publication-detail-pages/statements/monetary-poli...


In [10]:
# Reload fresh from CSV to get original date values
df_raw = pd.read_csv(DATA_IN)

# Copy original dates into a backup column before any modification
df['date_original'] = df_raw['date']

def fix_sarb_date(d):
    d = str(d).strip()
    
    # Format 1 — already clean: "January 2026"
    if re.match(r'^(January|February|March|April|May|June|July|August|'
                r'September|October|November|December)\s+\d{4}$', d):
        return d, False
    
    # Format 2 — garbled title: "Statement Of The MPC July 2024 2024"
    m = re.search(r'(January|February|March|April|May|June|July|August|'
                  r'September|October|November|December)\s+(\d{4})', d)
    if m:
        return f"{m.group(1)} {m.group(2)}", False
    
    # Format 3 — numeric ID + year: "5726 2013" or "16 2011"
    # Pattern is always: [fragment_id] [4-digit-year]
    # Year is always a plausible year (2000–2030), fragment IDs are not
    m = re.search(r'\b(20\d{2})\b', d)
    if m:
        return f"January {m.group(1)}", True
    
    return None, True

# Apply to SARB rows using original dates
sarb_mask = df['central_bank'] == 'SARB'
df.loc[sarb_mask, ['date', 'date_approximate']] = [
    fix_sarb_date(d) for d in df_raw.loc[sarb_mask, 'date']
]
df['date_approximate'] = df['date_approximate'].fillna(False).infer_objects(copy=False)

In [11]:
print(df[df['central_bank'] == 'SARB'][['date', 'date_approximate']].to_string())

               date  date_approximate
51     January 2026             False
52    November 2025             False
53   September 2025             False
54        July 2025             False
55         May 2025             False
56       March 2025             False
57     January 2025             False
58   September 2024             False
59   September 2024             False
60        July 2024             False
61         May 2024             False
62     January 2013              True
63     January 2013              True
64     January 2012              True
65     January 2012              True
66     January 2012              True
67     January 2012              True
68     January 2012              True
69     January 2012              True
70     January 2011              True
71     January 2011              True
72     January 2011              True
73     January 2011              True
74     January 2011              True
75     January 2011              True
76     Janua

In [12]:
def parse_date(d):
    d = str(d).strip()
    formats = [
        '%B %Y',        # "January 2026" — SARB, approximated
        '%B %d, %Y',    # "February 06, 2026" — RBI
        '%d %B %Y',     # "13 February 2026" — CBR
        '%B %d %Y',     # "February 06 2026" — fallback
        '%Y-%m-%d',     # ISO format — just in case
    ]
    for fmt in formats:
        try:
            return pd.to_datetime(d, format=fmt)
        except:
            continue
    return pd.NaT

df['date'] = df['date'].apply(parse_date)

print("Date dtype:", df['date'].dtype)
print("\nNull dates:", df['date'].isna().sum())
print("\nDate range by bank:")
print(df.groupby('central_bank')['date'].agg(['min', 'max']))

Date dtype: datetime64[ns]

Null dates: 0

Date range by bank:
                    min        max
central_bank                      
CBR          2018-10-26 2026-02-13
PBOC         1996-09-10 2025-10-27
RBI          2016-04-05 2026-02-06
SARB         2006-01-01 2026-01-01


In [14]:
# ── RBI boilerplate stripping ──────────────────────────────
def strip_rbi_boilerplate(text, title):
    # Case 1: Scraped statements — title appears in text, strip up to and including it
    title_clean = title.strip()
    idx = text.find(title_clean)
    if idx != -1:
        return text[idx + len(title_clean):].strip()
    
    # Case 2: PDF-extracted statements — find where actual content starts
    m = re.search(r'\n(Assessment|Part A|Monetary and Liquidity|On the basis|The Monetary Policy Committee|Good morning|Good evening)', text)
    if m:
        return text[m.start():].strip()
    
    # Fallback — strip up to first "1." paragraph marker
    m = re.search(r'\n1\.', text)
    if m:
        return text[m.start():].strip()
    
    return text

# ── SARB boilerplate stripping ─────────────────────────────
def strip_sarb_boilerplate(text):
    m = re.search(r'Issued by .+?Governor of the South African Reserve Bank', text)
    if m:
        return text[m.end():].strip()
    # Fallback
    m = re.search(r'\n(Last |Good |The |In |South )', text)
    if m:
        return text[m.start():].strip()
    return text

# ── PBOC spacing fix ───────────────────────────────────────
def fix_pboc_spacing(text):
    return re.sub(r' {2,}', ' ', text)

# ── Apply per bank ─────────────────────────────────────────
def clean_boilerplate(row):
    bank = row['central_bank']
    text = str(row['text'])
    if bank == 'RBI':
        return strip_rbi_boilerplate(text, str(row['title']))
    elif bank == 'SARB':
        return strip_sarb_boilerplate(text)
    elif bank == 'PBOC':
        return fix_pboc_spacing(text)
    return text

df['text'] = df.apply(clean_boilerplate, axis=1)

# Verify — check first 500 chars per bank
for bank in ['RBI', 'CBR', 'SARB', 'PBOC']:
    print(f"\n{'='*60}")
    print(f"BANK: {bank}")
    print(df[df['central_bank'] == bank]['text'].iloc[0][:500])


BANK: RBI
Part A: Monetary Policy 
Monetary and Liquidity Measures  
 
On the basis of an assessment of the current and evolving macroeconomic 
situation, it has been decided to:  
• reduce the policy repo rate under the liquidity adjustment facility (LAF) by 25 
basis points from 6.75 per cent to 6.5 per cent;  
• reduce the minimum daily maintenance of the cash reserve ratio (CRR) from 
95 per cent of the requirement to 90 per cent with effect from the fortnight 
beginning April 16, 2016, while keeping 

BANK: CBR
On 13 February 2026, the Bank of Russia Board of Directors decided to cut the key rate by 50 basis points to 15.50% per annum. The economy continues to return to a balanced growth path. In January, price growth accelerated significantly due to one-off factors. However, the Bank of Russia estimates that the underlying measures of current price growth have not changed considerably. After the effect of one-off factors fades, disinflation will continue. The Bank of Russia will

In [15]:
# Look at 3 RBI samples to find a consistent stripping anchor
for i in range(3):
    row = df[df['central_bank'] == 'RBI'].iloc[i]
    print(f"\n--- RBI sample {i+1} ---")
    print(f"TITLE: {row['title']}")
    print(f"TEXT (first 300 chars):\n{row['text'][:300]}")
    print()


--- RBI sample 1 ---
TITLE: Governor's Statement: April 5, 2016
TEXT (first 300 chars):
Part A: Monetary Policy 
Monetary and Liquidity Measures  
 
On the basis of an assessment of the current and evolving macroeconomic 
situation, it has been decided to:  
• reduce the policy repo rate under the liquidity adjustment facility (LAF) by 25 
basis points from 6.75 per cent to 6.5 per cen


--- RBI sample 2 ---
TITLE: Governor's Statement: June 7, 2016
TEXT (first 300 chars):
Monetary and Liquidity Measures 
On the basis of an assessment of the current and evolving macroeconomic 
situation, it has been decided to: 
 keep the policy repo rate under the liquidity adjustment facility (LAF) 
unchanged at 6.5 per cent;  
 keep the cash reserve ratio (CRR) of scheduled banks


--- RBI sample 3 ---
TITLE: Governor's Statement: August 9, 2016
TEXT (first 300 chars):
Monetary and Liquidity Measures 
On the basis of an assessment of the current and evolving macroeconomic 
situation, it has been 

In [16]:
# Load spaCy model
nlp = spacy.load('en_core_web_sm')

# Add custom central banking stopwords on top of spaCy defaults
custom_stopwords = {
    'bank', 'central', 'committee', 'monetary', 'policy', 'reserve',
    'governor', 'statement', 'percent', 'basis', 'point', 'per', 'annum',
    'said', 'also', 'would', 'could', 'may', 'shall', 'will'
}

def clean_text(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # 3. Remove markdown headers
    text = re.sub(r'##+ ?', '', text)
    # 4. Replace hyphens and slashes with space (before punctuation removal)
    text = re.sub(r'[-/]', ' ', text)
    # 5. Remove standalone numbers
    text = re.sub(r'\b\d+\b', '', text)
    # 6. Remove punctuation and special characters
    text = re.sub(r'[^a-z\s]', '', text)
    # 7. Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # 8. spaCy — lemmatise and remove stopwords
    doc = nlp(text)
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop
        and token.is_alpha
        and token.lemma_ not in custom_stopwords
        and len(token.lemma_) > 2
    ]
    return ' '.join(tokens)

# Apply — this will take a minute
print("Cleaning texts...")
df['text_clean'] = df['text'].apply(clean_text)
print("Done.")
print(f"\nSample cleaned text (RBI):")
print(df[df['central_bank'] == 'RBI']['text_clean'].iloc[0][:500])

Cleaning texts...
Done.

Sample cleaned text (RBI):
liquidity measure assessment current evolve macroeconomic situation decide reduce repo rate liquidity adjustment facility laf cent cent reduce minimum daily maintenance cash ratio crr cent requirement cent effect fortnight beginning april keep crr unchanged cent net demand time liability ndtl continue provide liquidity require progressively low average ante liquidity deficit system cent ndtl position close neutrality narrow rate corridor bps bps reduce msf rate increase reverse repo rate view en


In [17]:
# Check cleaned text across all banks
for bank in ['RBI', 'CBR', 'SARB', 'PBOC']:
    sample = df[df['central_bank'] == bank]['text_clean'].iloc[0]
    print(f"\n{'='*60}")
    print(f"BANK: {bank}")
    print(f"Cleaned text (first 300 chars):\n{sample[:300]}")

# Check token counts
df['token_count'] = df['text_clean'].apply(lambda x: len(x.split()))
print(f"\n{'='*60}")
print("Token count statistics by bank:")
print(df.groupby('central_bank')['token_count'].describe().round(1))


BANK: RBI
Cleaned text (first 300 chars):
liquidity measure assessment current evolve macroeconomic situation decide reduce repo rate liquidity adjustment facility laf cent cent reduce minimum daily maintenance cash ratio crr cent requirement cent effect fortnight beginning april keep crr unchanged cent net demand time liability ndtl contin

BANK: CBR
Cleaned text (first 300 chars):
february russia board director decide cut key rate economy continue return balanced growth path january price growth accelerate significantly factor russia estimate underlie measure current price growth change considerably effect factor fade disinflation continue russia assess need key rate cut upco

BANK: SARB
Cleaned text (first 300 chars):
year mark extreme global uncertainty begin new round shock geopolitical tension remain elevated reflect appear rupture global political order new threat independence market jittery precious metal like gold receive safe haven flow ongoing risk artificial intelligence b

In [18]:
# Final checks before saving
print("=== Final dataset summary ===")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nNull values:\n{df.isnull().sum()}")
print(f"\nDate range overall: {df['date'].min()} to {df['date'].max()}")
print(f"\nApproximate dates: {df['date_approximate'].sum()} rows")

# Save
DATA_OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(DATA_OUT, index=False)
print(f"\nSaved to: {DATA_OUT}")

=== Final dataset summary ===
Shape: (302, 10)

Columns: ['country', 'central_bank', 'date', 'title', 'text', 'url', 'date_original', 'date_approximate', 'text_clean', 'token_count']

Null values:
country             0
central_bank        0
date                0
title               0
text                0
url                 0
date_original       0
date_approximate    0
text_clean          0
token_count         0
dtype: int64

Date range overall: 1996-09-10 00:00:00 to 2026-02-13 00:00:00

Approximate dates: 46 rows

Saved to: data/brics_mpc_cleaned.csv


## Output

The cleaned dataset has been saved to `data/brics_mpc_cleaned.csv`.

**Final dataset: 302 rows, 10 columns**

| Bank | Statements | Coverage |
|------|------------|----------|
| PBOC | 131 | 1996–2025 |
| RBI | 63 | 2016–2026 |
| SARB | 57 | 2006–2026 |
| CBR | 51 | 2018–2026 |

**New columns added:**
- `text_clean` — preprocessed text ready for sentiment scoring and LDA
- `token_count` — number of tokens in cleaned text
- `date_approximate` — `True` for 46 SARB rows where only year was recoverable
- `date_original` — original raw date string before parsing

**Known limitations to carry forward:**
- 46 SARB statements (2006–2013) have approximate dates (January placeholder)
- RBI PDF-extracted statements (2016–2025) use ISO date format from filename
- PBOC coverage is broader than other banks (1996–2025, varied communication types)
- RBI coverage starts 2016 — no pre-MPC baseline available